In [27]:

import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")
print("✅ Libraries loaded successfully.")

✅ Libraries loaded successfully.


In [28]:

from google.colab import files

uploaded = files.upload()

Saving Video_Games_Sales_as_at_22_Dec_2016.csv to Video_Games_Sales_as_at_22_Dec_2016 (1).csv


In [29]:
import os

RAW_PATH       = '/content/Video_Games_Sales_as_at_22_Dec_2016.csv'
PROCESSED_PATH = '/content/video_games_cleaned.csv'
LOG_PATH       = '/content/etl_step_log.txt'

print("✅ Paths configured.")

✅ Paths configured.


In [30]:

step_log = []

def log_step(step_number, description, df_before, df_after, extra_note=""):
    rows_removed = len(df_before) - len(df_after)
    entry = {
        "step"         : step_number,
        "description"  : description,
        "rows_before"  : len(df_before),
        "rows_after"   : len(df_after),
        "rows_removed" : rows_removed,
        "timestamp"    : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "note"         : extra_note
    }
    step_log.append(entry)
    print(f"[STEP {step_number}] {description}")
    print(f"         Rows before: {entry['rows_before']}  |  Rows after: {entry['rows_after']}  |  Removed: {rows_removed}")
    if extra_note:
        print(f"         Note: {extra_note}")
    print()

print("✅ Logger ready.")

✅ Logger ready.


In [31]:

df_raw = pd.read_csv(RAW_PATH, encoding="utf-8")
df = df_raw.copy()  # working copy — raw file is NEVER modified

print(f" Raw dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

 Raw dataset loaded: 16719 rows × 16 columns
Columns: ['Name', 'Platform', 'Year_of_Release', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales', 'Critic_Score', 'Critic_Count', 'User_Score', 'User_Count', 'Developer', 'Rating']


In [32]:

print("=" * 60)
print("INITIAL DATA PROFILE")
print("=" * 60)

print("\n── Shape ──")
print(f"  Rows: {df.shape[0]}   Columns: {df.shape[1]}")

print("\n── Data Types ──")
print(df.dtypes)

print("\n── Missing Values ──")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print(missing_df[missing_df["Missing Count"] > 0])

print("\n── Duplicate Rows ──")
print(f"  Duplicates: {df.duplicated().sum()}")

print("\n── Sample Rows ──")
df.head(3)

INITIAL DATA PROFILE

── Shape ──
  Rows: 16719   Columns: 16

── Data Types ──
Name                object
Platform            object
Year_of_Release    float64
Genre               object
Publisher           object
NA_Sales           float64
EU_Sales           float64
JP_Sales           float64
Other_Sales        float64
Global_Sales       float64
Critic_Score       float64
Critic_Count       float64
User_Score          object
User_Count         float64
Developer           object
Rating              object
dtype: object

── Missing Values ──
                 Missing Count  Missing %
Name                         2       0.01
Year_of_Release            269       1.61
Genre                        2       0.01
Publisher                   54       0.32
Critic_Score              8582      51.33
Critic_Count              8582      51.33
User_Score                6704      40.10
User_Count                9129      54.60
Developer                 6623      39.61
Rating                    6769  

,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E


In [33]:

df_before = df.copy()

df = df[df["Name"].notnull()].reset_index(drop=True)

log_step(1, "Drop rows where 'Name' is null", df_before, df,
         extra_note="Name is the primary record identifier — null names are unanalysable.")

[STEP 1] Drop rows where 'Name' is null
         Rows before: 16719  |  Rows after: 16717  |  Removed: 2
         Note: Name is the primary record identifier — null names are unanalysable.



In [34]:

df_before = df.copy()

# Replace 'N/A' string with NaN
df["Year_of_Release"] = df["Year_of_Release"].replace("N/A", np.nan)

# Convert to numeric
df["Year_of_Release"] = pd.to_numeric(df["Year_of_Release"], errors="coerce")

# Count nulls before fill
null_year_count = df["Year_of_Release"].isnull().sum()

# Fill nulls with median year
median_year = int(df["Year_of_Release"].median())
df["Year_of_Release"] = df["Year_of_Release"].fillna(median_year)

# Remove implausible years (before 1980 or after 2016)
df = df[(df["Year_of_Release"] >= 1980) & (df["Year_of_Release"] <= 2016)].reset_index(drop=True)

# Convert to integer
df["Year_of_Release"] = df["Year_of_Release"].astype(int)

log_step(2, "Fix Year_of_Release — convert type, fill nulls, cap range [1980–2016]",
         df_before, df,
         extra_note=f"Replaced {null_year_count} nulls with median year ({median_year}). Removed years outside 1980–2016.")

[STEP 2] Fix Year_of_Release — convert type, fill nulls, cap range [1980–2016]
         Rows before: 16717  |  Rows after: 16713  |  Removed: 4
         Note: Replaced 269 nulls with median year (2007). Removed years outside 1980–2016.



In [35]:

df_before = df.copy()

text_cols = ["Name", "Genre", "Publisher", "Developer", "Platform", "Rating"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Title case for Genre, Publisher, Developer only
for col in ["Genre", "Publisher", "Developer"]:
    df[col] = df[col].str.title()

# Normalise common 'Unknown' variants created by astype(str) on NaN
for col in text_cols:
    df[col] = df[col].replace({"Nan": "Unknown", "nan": "Unknown", "None": "Unknown", "": "Unknown"})

log_step(3, "Standardise text columns — strip whitespace, title case, normalise null strings",
         df_before, df,
         extra_note=f"Applied to columns: {text_cols}")

[STEP 3] Standardise text columns — strip whitespace, title case, normalise null strings
         Rows before: 16713  |  Rows after: 16713  |  Removed: 0
         Note: Applied to columns: ['Name', 'Genre', 'Publisher', 'Developer', 'Platform', 'Rating']



In [36]:

df_before = df.copy()

df["User_Score"] = df["User_Score"].replace("tbd", np.nan)
df["User_Score"] = pd.to_numeric(df["User_Score"], errors="coerce")

tbd_count = df_before["User_Score"].astype(str).str.lower().eq("tbd").sum()
null_before = df["User_Score"].isnull().sum()

median_user = df["User_Score"].median()
df["User_Score"] = df["User_Score"].fillna(median_user)

log_step(4, "Clean User_Score — replace 'tbd', convert to float, fill nulls with median",
         df_before, df,
         extra_note=f"'tbd' entries: {tbd_count}. Nulls filled with median ({median_user:.2f}).")

[STEP 4] Clean User_Score — replace 'tbd', convert to float, fill nulls with median
         Rows before: 16713  |  Rows after: 16713  |  Removed: 0
         Note: 'tbd' entries: 2424. Nulls filled with median (7.50).



In [37]:

df_before = df.copy()

for col in ["Critic_Score", "Critic_Count", "User_Count"]:
    null_count = df[col].isnull().sum()
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"  {col}: {null_count} nulls filled with median ({median_val:.1f})")

log_step(5, "Fill nulls in Critic_Score, Critic_Count, User_Count with column medians",
         df_before, df)

  Critic_Score: 8576 nulls filled with median (71.0)
  Critic_Count: 8576 nulls filled with median (21.0)
  User_Count: 9123 nulls filled with median (24.0)
[STEP 5] Fill nulls in Critic_Score, Critic_Count, User_Count with column medians
         Rows before: 16713  |  Rows after: 16713  |  Removed: 0



In [38]:

df_before = df.copy()

sales_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]
for col in sales_cols:
    null_count = df[col].isnull().sum()
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    if null_count > 0:
        print(f"  {col}: {null_count} nulls set to 0")

# Recalculate Global_Sales for consistency
df["Global_Sales_Calculated"] = (
    df["NA_Sales"] + df["EU_Sales"] + df["JP_Sales"] + df["Other_Sales"]
).round(2)

log_step(6, "Null sales set to 0; Global_Sales_Calculated added as sum of regional sales",
         df_before, df,
         extra_note="Global_Sales_Calculated can be used to cross-validate original Global_Sales.")

[STEP 6] Null sales set to 0; Global_Sales_Calculated added as sum of regional sales
         Rows before: 16713  |  Rows after: 16713  |  Removed: 0
         Note: Global_Sales_Calculated can be used to cross-validate original Global_Sales.



In [39]:

df_before = df.copy()

df = df.drop_duplicates().reset_index(drop=True)

log_step(7, "Remove exact duplicate rows", df_before, df)

[STEP 7] Remove exact duplicate rows
         Rows before: 16713  |  Rows after: 16713  |  Removed: 0



In [40]:

df_before = df.copy()

df["Critic_Score"]  = df["Critic_Score"].astype(float)
df["Critic_Count"]  = df["Critic_Count"].astype(int)
df["User_Count"]    = df["User_Count"].astype(int)
df["User_Score"]    = df["User_Score"].astype(float)
df["Year_of_Release"] = df["Year_of_Release"].astype(int)

log_step(8, "Cast columns to correct final data types", df_before, df,
         extra_note="Critic_Count, User_Count → int; Scores → float; Year → int.")

[STEP 8] Cast columns to correct final data types
         Rows before: 16713  |  Rows after: 16713  |  Removed: 0
         Note: Critic_Count, User_Count → int; Scores → float; Year → int.



In [41]:

df_before = df.copy()

# Sales_Region_Dominant
region_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
df["Sales_Region_Dominant"] = df[region_cols].idxmax(axis=1).str.replace("_Sales", "")

# Score_Gap (Critic scores are 0–100; User scores are 0–10 → scale up)
df["Score_Gap"] = (df["Critic_Score"] - (df["User_Score"] * 10)).round(2)

# Era
def get_era(year):
    if year < 1990: return "1980s"
    elif year < 2000: return "1990s"
    elif year < 2010: return "2000s"
    else: return "2010s"

df["Era"] = df["Year_of_Release"].apply(get_era)

log_step(9, "Add derived columns: Sales_Region_Dominant, Score_Gap, Era",
         df_before, df,
         extra_note="Feature engineering for EDA and Tableau dashboard use.")

[STEP 9] Add derived columns: Sales_Region_Dominant, Score_Gap, Era
         Rows before: 16713  |  Rows after: 16713  |  Removed: 0
         Note: Feature engineering for EDA and Tableau dashboard use.



In [42]:

df_before = df.copy()

print("── Final Null Check ──")
final_nulls = df.isnull().sum()
print(final_nulls[final_nulls > 0] if final_nulls.sum() > 0 else "✅ No nulls remaining.")

# Reorder columns logically
col_order = [
    "Name", "Platform", "Year_of_Release", "Era", "Genre", "Publisher", "Developer", "Rating",
    "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales", "Global_Sales_Calculated",
    "Sales_Region_Dominant",
    "Critic_Score", "Critic_Count", "User_Score", "User_Count", "Score_Gap"
]
df = df[col_order]

log_step(10, "Final null check and column reorder", df_before, df)

── Final Null Check ──
✅ No nulls remaining.
[STEP 10] Final null check and column reorder
         Rows before: 16713  |  Rows after: 16713  |  Removed: 0



In [43]:

print("=" * 60)
print("POST-CLEANING DATA PROFILE")
print("=" * 60)

print(f"\n  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n── Data Types ──")
print(df.dtypes)
print(f"\n── Sample (cleaned) ──")
df.head()

POST-CLEANING DATA PROFILE

  Shape: 16713 rows × 20 columns

── Data Types ──
Name                        object
Platform                    object
Year_of_Release              int64
Era                         object
Genre                       object
Publisher                   object
Developer                   object
Rating                      object
NA_Sales                   float64
EU_Sales                   float64
JP_Sales                   float64
Other_Sales                float64
Global_Sales               float64
Global_Sales_Calculated    float64
Sales_Region_Dominant       object
Critic_Score               float64
Critic_Count                 int64
User_Score                 float64
User_Count                   int64
Score_Gap                  float64
dtype: object

── Sample (cleaned) ──


,Name,Platform,Year_of_Release,Era,Genre,Publisher,Developer,Rating,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Global_Sales_Calculated,Sales_Region_Dominant,Critic_Score,Critic_Count,User_Score,User_Count,Score_Gap
0,Wii Sports,Wii,2006,2000s,Sports,Nintendo,Nintendo,E,41.36,28.96,3.77,8.45,82.53,82.54,NA,76.0,51,8.0,322,-4.0
1,Super Mario Bros.,NES,1985,1980s,Platform,Nintendo,Unknown,Unknown,29.08,3.58,6.81,0.77,40.24,40.24,NA,71.0,21,7.5,24,-4.0
2,Mario Kart Wii,Wii,2008,2000s,Racing,Nintendo,Nintendo,E,15.68,12.76,3.79,3.29,35.52,35.52,NA,82.0,73,8.3,709,-1.0
3,Wii Sports Resort,Wii,2009,2000s,Sports,Nintendo,Nintendo,E,15.61,10.93,3.28,2.95,32.77,32.77,NA,80.0,73,8.0,192,0.0
4,Pokemon Red/Pokemon Blue,GB,1996,1990s,Role-Playing,Nintendo,Unknown,Unknown,11.27,8.89,10.22,1.00,31.37,31.38,NA,71.0,21,7.5,24,-4.0


In [44]:

df.to_csv(PROCESSED_PATH, index=False)
print(f"✅ Cleaned dataset saved → {PROCESSED_PATH}")
print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")

✅ Cleaned dataset saved → /content/video_games_cleaned.csv
   Shape: 16713 rows × 20 columns


In [45]:

log_df = pd.DataFrame(step_log)

with open(LOG_PATH, "w") as f:
    f.write("ETL STEP LOG — 02_cleaning.ipynb\n")
    f.write(f"Run Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 70 + "\n\n")
    for _, row in log_df.iterrows():
        f.write(f"STEP {row['step']}: {row['description']}\n")
        f.write(f"  Timestamp    : {row['timestamp']}\n")
        f.write(f"  Rows Before  : {row['rows_before']}\n")
        f.write(f"  Rows After   : {row['rows_after']}\n")
        f.write(f"  Rows Removed : {row['rows_removed']}\n")
        if row["note"]:
            f.write(f"  Note         : {row['note']}\n")
        f.write("\n")

print(f"✅ ETL Step Log saved → {LOG_PATH}")

# Print summary table
print("\n── Step Log Summary ──")
print(log_df[["step", "description", "rows_before", "rows_after", "rows_removed"]].to_string(index=False))

✅ ETL Step Log saved → /content/etl_step_log.txt

── Step Log Summary ──
 step                                                                     description  rows_before  rows_after  rows_removed
    1                                                  Drop rows where 'Name' is null        16719       16717             2
    2           Fix Year_of_Release — convert type, fill nulls, cap range [1980–2016]        16717       16713             4
    3 Standardise text columns — strip whitespace, title case, normalise null strings        16713       16713             0
    4      Clean User_Score — replace 'tbd', convert to float, fill nulls with median        16713       16713             0
    5        Fill nulls in Critic_Score, Critic_Count, User_Count with column medians        16713       16713             0
    6     Null sales set to 0; Global_Sales_Calculated added as sum of regional sales        16713       16713             0
    7                                               

In [46]:

print("=" * 60)
print("ETL PIPELINE COMPLETE")
print("=" * 60)
print(f"  Raw rows          : {len(df_raw)}")
print(f"  Cleaned rows      : {len(df)}")
print(f"  Total removed     : {len(df_raw) - len(df)}")
print(f"  Final columns     : {df.shape[1]}")
print(f"  Cleaned file      : {PROCESSED_PATH}")
print(f"  Step log          : {LOG_PATH}")
print("\n✅ Ready for notebooks/03_eda.ipynb")

ETL PIPELINE COMPLETE
  Raw rows          : 16719
  Cleaned rows      : 16713
  Total removed     : 6
  Final columns     : 20
  Cleaned file      : /content/video_games_cleaned.csv
  Step log          : /content/etl_step_log.txt

✅ Ready for notebooks/03_eda.ipynb


In [47]:

from google.colab import files

files.download(PROCESSED_PATH)   # downloads cleaned CSV
files.download(LOG_PATH)          # downloads step log

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>